# JJA

In [138]:
"""
Climate Network Analysis - Granger Causality and SHAP Dependencies
Visualizes relationships between climate variables using network graphs
Combined visualization with side-by-side subplots
"""

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Tuple, Optional


# Configuration
VARIABLE_MAPPING = {
    'temperature': 'T',
    'precipitable_water': 'Prw',
    'precipitation': 'Pr',
    'evaporation': 'E',
    'soil_moisture': 'SM',
    'runoff': 'q'
}

VARIABLE_COLORS_LIGHT = {
    'T': '#FF8A80',
    'Prw': '#81C7E5',
    'Pr': '#A5D6A7',
    'E': '#F8BBD9',
    'SM': '#B39DDB',
    'q': '#D7CCC8'
}

VARIABLE_COLORS_DARK = {
    'T': '#E53935',
    'Prw': '#1976D2',
    'Pr': '#388E3C',
    'E': '#E91E63',
    'SM': '#7B1FA2',
    'q': '#5D4037'
}

FIXED_POSITIONS = {
    'T': (0.0, 1.0),
    'Prw': (0.866, 0.5),
    'Pr': (0.866, -0.5),
    'E': (0.0, -1.0),
    'SM': (-0.866, -0.5),
    'q': (-0.866, 0.5)
}


def load_data(filepath: str) -> Optional[pd.DataFrame]:
    """Load combined Granger and SHAP data from CSV file."""
    try:
        df = pd.read_csv(filepath)
        # Convert boolean strings to actual booleans
        if df['granger_significant'].dtype == object:
            df['granger_significant'] = df['granger_significant'].map({'True': True, 'False': False})
        
        print(f"✓ Successfully loaded data: {len(df)} relationships")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Granger significant: {df['granger_significant'].sum()}")
        print(f"  SHAP dependencies ≥ 0.01: {(df['shap_dependency'].abs() >= 0.01).sum()}")
        return df
    except FileNotFoundError:
        print(f"❌ File not found: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None


def parse_granger_edges(df: pd.DataFrame, p_threshold: float = 0.05) -> List[Dict]:
    """Extract significant Granger causality relationships."""
    edges = []
    
    for _, row in df[df['granger_significant']].iterrows():
        source = VARIABLE_MAPPING.get(row['source_variable'], row['source_variable'])
        target = VARIABLE_MAPPING.get(row['target_variable'], row['target_variable'])
        
        # Skip self-loops
        if source == target:
            continue
        
        p_value = row['granger_p_value']
        edges.append({
            'source': source,
            'target': target,
            'p_value': p_value,
            'lag': row['granger_lag'],
            'weight': -np.log10(p_value) if p_value > 0 else 10,
            'type': 'granger'
        })
    
    return edges


def parse_shap_edges(df: pd.DataFrame, dependency_threshold: float = 0.001) -> List[Dict]:
    """Extract significant SHAP dependency relationships."""
    edges = []
    
    for _, row in df.iterrows():
        source = VARIABLE_MAPPING.get(row['source_variable'], row['source_variable'])
        target = VARIABLE_MAPPING.get(row['target_variable'], row['target_variable'])
        
        # Skip self-loops
        if source == target:
            continue
        
        dependency = abs(row['shap_dependency'])
        if dependency >= dependency_threshold:
            edges.append({
                'source': source,
                'target': target,
                'dependency': dependency,
                'direction': row['shap_direction'],
                'weight': dependency * 100,
                'type': 'shap'
            })
    
    return edges


def create_network(edges: List[Dict]) -> nx.DiGraph:
    """Create NetworkX directed graph from edge data."""
    G = nx.DiGraph()
    
    for edge in edges:
        if edge['type'] == 'granger':
            G.add_edge(
                edge['source'],
                edge['target'],
                weight=edge['weight'],
                p_value=edge['p_value'],
                lag=edge['lag'],
                edge_type='granger'
            )
        else:  # SHAP
            G.add_edge(
                edge['source'],
                edge['target'],
                weight=edge['weight'],
                dependency=edge['dependency'],
                direction=edge['direction'],
                edge_type='shap'
            )
    
    return G


def analyze_network(G: nx.DiGraph, analysis_type: str, title: str = "Network") -> None:
    """Print network statistics and properties."""
    print(f"\n{analysis_type.upper()} NETWORK ANALYSIS - {title}")
    print("=" * 60)
    print(f"Variables: {G.number_of_nodes()}")
    print(f"Relationships: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.3f}")
    
    if G.number_of_edges() == 0:
        print("No significant relationships found.")
        return
    
    # In-degree (most influenced)
    in_degrees = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)
    print("\nMost influenced variables (incoming relationships):")
    for node, degree in in_degrees:
        if degree > 0:
            print(f"  {node}: {degree}")
    
    # Out-degree (most influential)
    out_degrees = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)
    print("\nMost influential variables (outgoing relationships):")
    for node, degree in out_degrees:
        if degree > 0:
            print(f"  {node}: {degree}")
    
    # Top relationships
    if analysis_type == 'granger':
        edges = [(u, v, G[u][v]['p_value'], G[u][v]['lag']) for u, v in G.edges()]
        edges.sort(key=lambda x: x[2])
        
        print("\nStrongest causal relationships (lowest p-values):")
        for i, (u, v, p_val, lag) in enumerate(edges[:5], 1):
            print(f"  {i}. {u} → {v}: p={p_val:.2e}, lag={lag}")
    
    elif analysis_type == 'shap':
        edges = [(u, v, G[u][v]['dependency'], G[u][v]['direction']) for u, v in G.edges()]
        edges.sort(key=lambda x: x[2], reverse=True)
        
        print("\nStrongest dependency relationships:")
        for i, (u, v, dep, direction) in enumerate(edges[:5], 1):
            direction_str = "positive" if direction > 0 else "negative"
            print(f"  {i}. {u} → {v}: dep={dep:.4f}, dir={direction:.4f} ({direction_str})")
        
        # Direction distribution
        positive = sum(1 for u, v in G.edges() if G[u][v]['direction'] > 0)
        negative = sum(1 for u, v in G.edges() if G[u][v]['direction'] < 0)
        print(f"\nSHAP direction distribution:")
        print(f"  Positive: {positive} relationships")
        print(f"  Negative: {negative} relationships")


def plot_combined_networks(granger_G: nx.DiGraph, shap_G: nx.DiGraph, 
                          model_name: str, region: str,
                          figsize: Tuple = (16, 8), save_path: Optional[str] = None) -> None:
    """Plot both Granger and SHAP networks side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, dpi=800)
    
    from matplotlib.lines import Line2D
    
    # ==================== SUBPLOT 1: GRANGER CAUSALITY ====================
    if granger_G and granger_G.number_of_edges() > 0:
        # Draw edges
        for u, v in granger_G.edges():
            source_color = VARIABLE_COLORS_LIGHT.get(u, '#CCCCCC')
            p_value = granger_G[u][v]['p_value']
            
            # Edge width based on significance
            if p_value < 0.001:
                width = 6.0
                alpha = 1.0
            elif p_value < 0.01:
                width = 3.0
                alpha = 1.0
            elif p_value < 0.05:
                width = 0.8
                alpha = 1.0
            else:
                width = 0

            # Skip drawing if width is 0
            if width == 0:
                continue

            nx.draw_networkx_edges(
                granger_G, FIXED_POSITIONS, 
                edgelist=[(u, v)],
                width=width, 
                edge_color=[source_color],
                arrows=True, 
                arrowsize=60, 
                arrowstyle='-|>',
                alpha=alpha, 
                connectionstyle="arc3,rad=0.1",
                min_source_margin=22, 
                min_target_margin=22, 
                ax=ax1
            )
        
        # Draw nodes
        node_colors = [VARIABLE_COLORS_DARK.get(node, '#666666') for node in granger_G.nodes()]
        nx.draw_networkx_nodes(
            granger_G, FIXED_POSITIONS, 
            node_color=node_colors, 
            node_size=1700,
            alpha=1.0, 
            edgecolors='#333333', 
            linewidths=2, 
            ax=ax1
        )
        
        # Draw labels
        nx.draw_networkx_labels(
            granger_G, FIXED_POSITIONS, 
            font_size=18, 
            font_weight='bold',
            font_color='white', 
            font_family='Arial', 
            ax=ax1
        )

        # Add legend for Granger
        legend_elements_granger = [
            Line2D([0], [0], color='gray', linewidth=6, label='Strong (p < 0.001)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=3, label='Moderate (p < 0.01)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=0.8, label='Weak (p < 0.05)', alpha=1.0),
        ]
        ax1.legend(handles=legend_elements_granger, loc='lower center', fontsize=16, 
                  frameon=True, fancybox=True, shadow=True, title='',
                  ncol=2, bbox_to_anchor=(0.5, -0.15))
    
    ax1.set_title('(i) Granger Causality', fontsize=18, fontweight='bold', pad=20)
    ax1.axis('off')
    
    # ==================== SUBPLOT 2: SHAP DEPENDENCIES ====================
    if shap_G and shap_G.number_of_edges() > 0:
        # Separate edges by direction
        positive_edges = [(u, v) for u, v in shap_G.edges() if shap_G[u][v]['direction'] > 0]
        negative_edges = [(u, v) for u, v in shap_G.edges() if shap_G[u][v]['direction'] < 0]
        
        def draw_shap_edges(edge_list, line_style):
            """Helper to draw edges with given style."""
            for u, v in edge_list:
                dependency = shap_G[u][v]['dependency']
                source_color = VARIABLE_COLORS_LIGHT.get(u, '#CCCCCC')
                
                # Width based on dependency strength
                if dependency > 0.03:
                    width = 6.0
                    alpha = 1.0
                elif dependency > 0.02:
                    width = 3
                    alpha = 1.0
                elif dependency > 0.015:
                    width = 0.8
                    alpha = 1.0   
                else:
                    width = 0
                
                # Skip drawing if width is 0
                if width == 0:
                    continue
                
                alpha = min(0.9, max(0.4, dependency * 20))
                
                nx.draw_networkx_edges(
                    shap_G, FIXED_POSITIONS,
                    edgelist=[(u, v)],
                    width=width,
                    edge_color=[source_color],
                    arrows=True,
                    arrowsize=60,
                    arrowstyle='-|>',
                    alpha=alpha,
                    connectionstyle="arc3,rad=0.1",
                    style=line_style,
                    min_source_margin=22,
                    min_target_margin=22,
                    ax=ax2
                )
        
        # Draw positive (solid) and negative (dashed) edges
        draw_shap_edges(positive_edges, '-')
        draw_shap_edges(negative_edges, '--')
        
        # Draw nodes
        node_colors = [VARIABLE_COLORS_DARK.get(node, '#666666') for node in shap_G.nodes()]
        nx.draw_networkx_nodes(
            shap_G, FIXED_POSITIONS,
            node_color=node_colors,
            node_size=1700,
            alpha=1.0,
            edgecolors='#333333',
            linewidths=2.5,
            ax=ax2
        )
        
        # Draw labels
        nx.draw_networkx_labels(
            shap_G, FIXED_POSITIONS,
            font_size=20,
            font_weight='bold',
            font_color='white',
            font_family='Arial',
            ax=ax2
        )

        # Add legend for SHAP
        legend_elements_shap = [
            Line2D([0], [0], color='gray', linewidth=6, label='Strong (Dv > 0.03)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=3, label='Moderate (Dv > 0.02)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=0.8, label='Weak (Dv > 0.015)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=2, linestyle='-', label='Upward', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=2, linestyle='--', label='Downward', alpha=1.0),
        ]
        ax2.legend(handles=legend_elements_shap, loc='lower center', fontsize=16,
                  frameon=True, fancybox=True, shadow=True, title='',
                  ncol=3, bbox_to_anchor=(0.5, -0.15), columnspacing=1.0)
    
    ax2.set_title('(ii) RF - SHAP', fontsize=20, fontweight='bold', pad=20)
    ax2.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=800, bbox_inches='tight', facecolor='white')
    
    plt.show()


def run_analysis(filepath: str, model_name: str, region: str, 
                 save_path: Optional[str] = None) -> Tuple[Optional[nx.DiGraph], Optional[nx.DiGraph]]:
    """
    Run complete network analysis pipeline.
    
    Args:
        filepath: Path to CSV file with Granger and SHAP results
        model_name: Climate model name for labeling
        region: Geographic region for labeling
        save_path: Optional path to save the combined network plot
        
    Returns:
        Tuple of (granger_network, shap_network)
    """
    print("🌍 CLIMATE NETWORK ANALYSIS")
    print("=" * 80)
    
    # Load data
    df = load_data(filepath)
    if df is None:
        return None, None
    
    # Parse relationships
    print("\n📊 Processing Granger causality...")
    granger_edges = parse_granger_edges(df)
    print(f"  Found {len(granger_edges)} significant relationships")
    
    print("\n🧠 Processing SHAP dependencies...")
    shap_edges = parse_shap_edges(df)
    print(f"  Found {len(shap_edges)} significant relationships")
    
    # Create networks
    granger_G = create_network(granger_edges) if granger_edges else None
    shap_G = create_network(shap_edges) if shap_edges else None
    
    # Analyze networks
    if granger_G:
        analyze_network(granger_G, 'granger', f"{model_name} ({region})")
    
    if shap_G:
        analyze_network(shap_G, 'shap', f"{model_name} ({region})")
    
    # Plot combined networks
    if granger_G or shap_G:
        output_path = save_path if save_path else "combined_network.png"
        plot_combined_networks(granger_G, shap_G, model_name, region, 
                              save_path=output_path)
    
    # Summary
    print("\n✅ Generated file:")
    print(f"  - {output_path}")
    
    print("\n" + "=" * 80)
    print("INTERPRETATION GUIDE:")
    print("=" * 80)
    print("📊 GRANGER CAUSALITY (Left Panel):")
    print("  • Edge thickness → Statistical significance (p-value)")
    print("  • Edge colors → Source variable identity")
    print("  • Shows temporal causal relationships")
    print("\n🧠 SHAP DEPENDENCIES (Right Panel):")
    print("  • Edge thickness → Dependency strength")
    print("  • Solid lines → Upward effect on target")
    print("  • Dashed lines → Downward effect on target")
    print("  • Shows feature importance relationships")
    
    return granger_G, shap_G


if __name__ == "__main__":
    # ==============================================================================
    # USAGE EXAMPLES - Customize these parameters for your analysis
    # ==============================================================================
    
    # Example 1: Basic usage with default output filename
    print("\n" + "="*80)
    print("EXAMPLE 1: Basic Analysis")
    print("="*80)
    granger_net, shap_net = run_analysis(
        filepath="jja_shap_granger_comparison.csv",
        #filepath="jja_shap_granger_comparison_2071-2100_ESM2_NSZ5.csv",
        model_name="NorESM2-MM",
        region="Subtropics_N"
    )
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    print("\nTo customize:")
    print("1. Edit the filepath to your CSV file")
    print("2. Change model_name and region as needed")
    print("3. Uncomment examples 2-4 for batch processing")
    print("4. Modify VARIABLE_MAPPING and colors as needed")


EXAMPLE 1: Basic Analysis
🌍 CLIMATE NETWORK ANALYSIS
✓ Successfully loaded data: 32 relationships
  Columns: ['target_variable', 'source_variable', 'shap_dependency', 'shap_direction', 'shap_interpretation', 'granger_p_value', 'granger_significant', 'granger_lag', 'agreement_category']
  Granger significant: 2
  SHAP dependencies ≥ 0.01: 23

📊 Processing Granger causality...
  Found 2 significant relationships

🧠 Processing SHAP dependencies...
  Found 29 significant relationships

GRANGER NETWORK ANALYSIS - NorESM2-MM (Subtropics_N)
Variables: 4
Relationships: 2
Density: 0.167

Most influenced variables (incoming relationships):
  E: 1
  Prw: 1

Most influential variables (outgoing relationships):
  Pr: 1
  SM: 1

Strongest causal relationships (lowest p-values):
  1. Pr → E: p=2.44e-02, lag=2
  2. SM → Prw: p=3.60e-02, lag=2

SHAP NETWORK ANALYSIS - NorESM2-MM (Subtropics_N)
Variables: 6
Relationships: 29
Density: 0.967

Most influenced variables (incoming relationships):
  Pr: 5
  


✅ Generated file:
  - combined_network.png

INTERPRETATION GUIDE:
📊 GRANGER CAUSALITY (Left Panel):
  • Edge thickness → Statistical significance (p-value)
  • Edge colors → Source variable identity
  • Shows temporal causal relationships

🧠 SHAP DEPENDENCIES (Right Panel):
  • Edge thickness → Dependency strength
  • Solid lines → Upward effect on target
  • Dashed lines → Downward effect on target
  • Shows feature importance relationships

✅ ANALYSIS COMPLETE!

To customize:
1. Edit the filepath to your CSV file
2. Change model_name and region as needed
3. Uncomment examples 2-4 for batch processing
4. Modify VARIABLE_MAPPING and colors as needed


# DJF

In [91]:
"""
Climate Network Analysis - Granger Causality and SHAP Dependencies
Visualizes relationships between climate variables using network graphs
Combined visualization with side-by-side subplots
"""

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Tuple, Optional


# Configuration
VARIABLE_MAPPING = {
    'temperature': 'T',
    'precipitable_water': 'Prw',
    'precipitation': 'Pr',
    'evaporation': 'E',
    'soil_moisture': 'SM',
    'runoff': 'q'
}

VARIABLE_COLORS_LIGHT = {
    'T': '#FF8A80',
    'Prw': '#81C7E5',
    'Pr': '#A5D6A7',
    'E': '#F8BBD9',
    'SM': '#B39DDB',
    'q': '#D7CCC8'
}

VARIABLE_COLORS_DARK = {
    'T': '#E53935',
    'Prw': '#1976D2',
    'Pr': '#388E3C',
    'E': '#E91E63',
    'SM': '#7B1FA2',
    'q': '#5D4037'
}

FIXED_POSITIONS = {
    'T': (0.0, 1.0),
    'Prw': (0.866, 0.5),
    'Pr': (0.866, -0.5),
    'E': (0.0, -1.0),
    'SM': (-0.866, -0.5),
    'q': (-0.866, 0.5)
}


def load_data(filepath: str) -> Optional[pd.DataFrame]:
    """Load combined Granger and SHAP data from CSV file."""
    try:
        df = pd.read_csv(filepath)
        # Convert boolean strings to actual booleans
        if df['granger_significant'].dtype == object:
            df['granger_significant'] = df['granger_significant'].map({'True': True, 'False': False})
        
        print(f"✓ Successfully loaded data: {len(df)} relationships")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Granger significant: {df['granger_significant'].sum()}")
        print(f"  SHAP dependencies ≥ 0.01: {(df['shap_dependency'].abs() >= 0.01).sum()}")
        return df
    except FileNotFoundError:
        print(f"❌ File not found: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None


def parse_granger_edges(df: pd.DataFrame, p_threshold: float = 0.05) -> List[Dict]:
    """Extract significant Granger causality relationships."""
    edges = []
    
    for _, row in df[df['granger_significant']].iterrows():
        source = VARIABLE_MAPPING.get(row['source_variable'], row['source_variable'])
        target = VARIABLE_MAPPING.get(row['target_variable'], row['target_variable'])
        
        # Skip self-loops
        if source == target:
            continue
        
        p_value = row['granger_p_value']
        edges.append({
            'source': source,
            'target': target,
            'p_value': p_value,
            'lag': row['granger_lag'],
            'weight': -np.log10(p_value) if p_value > 0 else 10,
            'type': 'granger'
        })
    
    return edges


def parse_shap_edges(df: pd.DataFrame, dependency_threshold: float = 0.001) -> List[Dict]:
    """Extract significant SHAP dependency relationships."""
    edges = []
    
    for _, row in df.iterrows():
        source = VARIABLE_MAPPING.get(row['source_variable'], row['source_variable'])
        target = VARIABLE_MAPPING.get(row['target_variable'], row['target_variable'])
        
        # Skip self-loops
        if source == target:
            continue
        
        dependency = abs(row['shap_dependency'])
        if dependency >= dependency_threshold:
            edges.append({
                'source': source,
                'target': target,
                'dependency': dependency,
                'direction': row['shap_direction'],
                'weight': dependency * 100,
                'type': 'shap'
            })
    
    return edges


def create_network(edges: List[Dict]) -> nx.DiGraph:
    """Create NetworkX directed graph from edge data."""
    G = nx.DiGraph()
    
    for edge in edges:
        if edge['type'] == 'granger':
            G.add_edge(
                edge['source'],
                edge['target'],
                weight=edge['weight'],
                p_value=edge['p_value'],
                lag=edge['lag'],
                edge_type='granger'
            )
        else:  # SHAP
            G.add_edge(
                edge['source'],
                edge['target'],
                weight=edge['weight'],
                dependency=edge['dependency'],
                direction=edge['direction'],
                edge_type='shap'
            )
    
    return G


def analyze_network(G: nx.DiGraph, analysis_type: str, title: str = "Network") -> None:
    """Print network statistics and properties."""
    print(f"\n{analysis_type.upper()} NETWORK ANALYSIS - {title}")
    print("=" * 60)
    print(f"Variables: {G.number_of_nodes()}")
    print(f"Relationships: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.3f}")
    
    if G.number_of_edges() == 0:
        print("No significant relationships found.")
        return
    
    # In-degree (most influenced)
    in_degrees = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)
    print("\nMost influenced variables (incoming relationships):")
    for node, degree in in_degrees:
        if degree > 0:
            print(f"  {node}: {degree}")
    
    # Out-degree (most influential)
    out_degrees = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)
    print("\nMost influential variables (outgoing relationships):")
    for node, degree in out_degrees:
        if degree > 0:
            print(f"  {node}: {degree}")
    
    # Top relationships
    if analysis_type == 'granger':
        edges = [(u, v, G[u][v]['p_value'], G[u][v]['lag']) for u, v in G.edges()]
        edges.sort(key=lambda x: x[2])
        
        print("\nStrongest causal relationships (lowest p-values):")
        for i, (u, v, p_val, lag) in enumerate(edges[:5], 1):
            print(f"  {i}. {u} → {v}: p={p_val:.2e}, lag={lag}")
    
    elif analysis_type == 'shap':
        edges = [(u, v, G[u][v]['dependency'], G[u][v]['direction']) for u, v in G.edges()]
        edges.sort(key=lambda x: x[2], reverse=True)
        
        print("\nStrongest dependency relationships:")
        for i, (u, v, dep, direction) in enumerate(edges[:5], 1):
            direction_str = "positive" if direction > 0 else "negative"
            print(f"  {i}. {u} → {v}: dep={dep:.4f}, dir={direction:.4f} ({direction_str})")
        
        # Direction distribution
        positive = sum(1 for u, v in G.edges() if G[u][v]['direction'] > 0)
        negative = sum(1 for u, v in G.edges() if G[u][v]['direction'] < 0)
        print(f"\nSHAP direction distribution:")
        print(f"  Positive: {positive} relationships")
        print(f"  Negative: {negative} relationships")


def plot_combined_networks(granger_G: nx.DiGraph, shap_G: nx.DiGraph, 
                          model_name: str, region: str,
                          figsize: Tuple = (16, 8), save_path: Optional[str] = None) -> None:
    """Plot both Granger and SHAP networks side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, dpi=800)
    
    from matplotlib.lines import Line2D
    
    # ==================== SUBPLOT 1: GRANGER CAUSALITY ====================
    if granger_G and granger_G.number_of_edges() > 0:
        # Draw edges
        for u, v in granger_G.edges():
            source_color = VARIABLE_COLORS_LIGHT.get(u, '#CCCCCC')
            p_value = granger_G[u][v]['p_value']
            
            # Edge width based on significance
            if p_value < 0.001:
                width = 6.0
                alpha = 1.0
            elif p_value < 0.01:
                width = 3.0
                alpha = 1.0
            elif p_value < 0.05:
                width = 0.8
                alpha = 1.0
            else:
                width = 0

            # Skip drawing if width is 0
            if width == 0:
                continue

            nx.draw_networkx_edges(
                granger_G, FIXED_POSITIONS, 
                edgelist=[(u, v)],
                width=width, 
                edge_color=[source_color],
                arrows=True, 
                arrowsize=60, 
                arrowstyle='-|>',
                alpha=alpha, 
                connectionstyle="arc3,rad=0.1",
                min_source_margin=22, 
                min_target_margin=22, 
                ax=ax1
            )
        
        # Draw nodes
        node_colors = [VARIABLE_COLORS_DARK.get(node, '#666666') for node in granger_G.nodes()]
        nx.draw_networkx_nodes(
            granger_G, FIXED_POSITIONS, 
            node_color=node_colors, 
            node_size=1700,
            alpha=1.0, 
            edgecolors='#333333', 
            linewidths=2, 
            ax=ax1
        )
        
        # Draw labels
        nx.draw_networkx_labels(
            granger_G, FIXED_POSITIONS, 
            font_size=18, 
            font_weight='bold',
            font_color='white', 
            font_family='Arial', 
            ax=ax1
        )

        # Add legend for Granger
        legend_elements_granger = [
            Line2D([0], [0], color='gray', linewidth=6, label='Strong (p < 0.001)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=3, label='Moderate (p < 0.01)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=0.8, label='Weak (p < 0.05)', alpha=1.0),
        ]
        ax1.legend(handles=legend_elements_granger, loc='lower center', fontsize=16, 
                  frameon=True, fancybox=True, shadow=True, title='',
                  ncol=2, bbox_to_anchor=(0.5, -0.15))
    
    ax1.set_title('(i) Granger Causality', fontsize=18, fontweight='bold', pad=20)
    ax1.axis('off')
    
    # ==================== SUBPLOT 2: SHAP DEPENDENCIES ====================
    if shap_G and shap_G.number_of_edges() > 0:
        # Separate edges by direction
        positive_edges = [(u, v) for u, v in shap_G.edges() if shap_G[u][v]['direction'] > 0]
        negative_edges = [(u, v) for u, v in shap_G.edges() if shap_G[u][v]['direction'] < 0]
        
        def draw_shap_edges(edge_list, line_style):
            """Helper to draw edges with given style."""
            for u, v in edge_list:
                dependency = shap_G[u][v]['dependency']
                source_color = VARIABLE_COLORS_LIGHT.get(u, '#CCCCCC')
                
                # Width based on dependency strength
                if dependency > 0.03:
                    width = 6.0
                    alpha = 1.0
                elif dependency > 0.02:
                    width = 3
                    alpha = 1.0
                elif dependency > 0.015:
                    width = 0.8
                    alpha = 1.0   
                else:
                    width = 0
                
                # Skip drawing if width is 0
                if width == 0:
                    continue
                
                alpha = min(0.9, max(0.4, dependency * 20))
                
                nx.draw_networkx_edges(
                    shap_G, FIXED_POSITIONS,
                    edgelist=[(u, v)],
                    width=width,
                    edge_color=[source_color],
                    arrows=True,
                    arrowsize=60,
                    arrowstyle='-|>',
                    alpha=alpha,
                    connectionstyle="arc3,rad=0.1",
                    style=line_style,
                    min_source_margin=22,
                    min_target_margin=22,
                    ax=ax2
                )
        
        # Draw positive (solid) and negative (dashed) edges
        draw_shap_edges(positive_edges, '-')
        draw_shap_edges(negative_edges, '--')
        
        # Draw nodes
        node_colors = [VARIABLE_COLORS_DARK.get(node, '#666666') for node in shap_G.nodes()]
        nx.draw_networkx_nodes(
            shap_G, FIXED_POSITIONS,
            node_color=node_colors,
            node_size=1700,
            alpha=1.0,
            edgecolors='#333333',
            linewidths=2.5,
            ax=ax2
        )
        
        # Draw labels
        nx.draw_networkx_labels(
            shap_G, FIXED_POSITIONS,
            font_size=20,
            font_weight='bold',
            font_color='white',
            font_family='Arial',
            ax=ax2
        )

        # Add legend for SHAP
        legend_elements_shap = [
            Line2D([0], [0], color='gray', linewidth=6, label='Strong (Dv > 0.03)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=3, label='Moderate (Dv > 0.02)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=0.8, label='Weak (Dv > 0.015)', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=2, linestyle='-', label='Upward', alpha=1.0),
            Line2D([0], [0], color='gray', linewidth=2, linestyle='--', label='Downward', alpha=1.0),
        ]
        ax2.legend(handles=legend_elements_shap, loc='lower center', fontsize=16,
                  frameon=True, fancybox=True, shadow=True, title='',
                  ncol=3, bbox_to_anchor=(0.5, -0.15), columnspacing=1.0)
    
    ax2.set_title('(ii) RF - SHAP', fontsize=20, fontweight='bold', pad=20)
    ax2.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=800, bbox_inches='tight', facecolor='white')
    
    plt.show()


def run_analysis(filepath: str, model_name: str, region: str, 
                 save_path: Optional[str] = None) -> Tuple[Optional[nx.DiGraph], Optional[nx.DiGraph]]:
    """
    Run complete network analysis pipeline.
    
    Args:
        filepath: Path to CSV file with Granger and SHAP results
        model_name: Climate model name for labeling
        region: Geographic region for labeling
        save_path: Optional path to save the combined network plot
        
    Returns:
        Tuple of (granger_network, shap_network)
    """
    print("🌍 CLIMATE NETWORK ANALYSIS")
    print("=" * 80)
    
    # Load data
    df = load_data(filepath)
    if df is None:
        return None, None
    
    # Parse relationships
    print("\n📊 Processing Granger causality...")
    granger_edges = parse_granger_edges(df)
    print(f"  Found {len(granger_edges)} significant relationships")
    
    print("\n🧠 Processing SHAP dependencies...")
    shap_edges = parse_shap_edges(df)
    print(f"  Found {len(shap_edges)} significant relationships")
    
    # Create networks
    granger_G = create_network(granger_edges) if granger_edges else None
    shap_G = create_network(shap_edges) if shap_edges else None
    
    # Analyze networks
    if granger_G:
        analyze_network(granger_G, 'granger', f"{model_name} ({region})")
    
    if shap_G:
        analyze_network(shap_G, 'shap', f"{model_name} ({region})")
    
    # Plot combined networks
    if granger_G or shap_G:
        output_path = save_path if save_path else "combined_network.png"
        plot_combined_networks(granger_G, shap_G, model_name, region, 
                              save_path=output_path)
    
    # Summary
    print("\n✅ Generated file:")
    print(f"  - {output_path}")
    
    print("\n" + "=" * 80)
    print("INTERPRETATION GUIDE:")
    print("=" * 80)
    print("📊 GRANGER CAUSALITY (Left Panel):")
    print("  • Edge thickness → Statistical significance (p-value)")
    print("  • Edge colors → Source variable identity")
    print("  • Shows temporal causal relationships")
    print("\n🧠 SHAP DEPENDENCIES (Right Panel):")
    print("  • Edge thickness → Dependency strength")
    print("  • Solid lines → Upward effect on target")
    print("  • Dashed lines → Downward effect on target")
    print("  • Shows feature importance relationships")
    
    return granger_G, shap_G


if __name__ == "__main__":
    # ==============================================================================
    # USAGE EXAMPLES - Customize these parameters for your analysis
    # ==============================================================================
    
    # Example 1: Basic usage with default output filename
    print("\n" + "="*80)
    print("EXAMPLE 1: Basic Analysis")
    print("="*80)
    granger_net, shap_net = run_analysis(
        #filepath="djf_shap_granger_comparison.csv",
        filepath="djf_shap_granger_comparison_2021-2050_CM2_SMZ.csv",
        model_name="NorESM2-MM",
        region="MidLatitude_N"
    )
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    print("\nTo customize:")
    print("1. Edit the filepath to your CSV file")
    print("2. Change model_name and region as needed")
    print("3. Uncomment examples 2-4 for batch processing")
    print("4. Modify VARIABLE_MAPPING and colors as needed")


EXAMPLE 1: Basic Analysis
🌍 CLIMATE NETWORK ANALYSIS
✓ Successfully loaded data: 32 relationships
  Columns: ['target_variable', 'source_variable', 'shap_dependency', 'shap_direction', 'shap_interpretation', 'granger_p_value', 'granger_significant', 'granger_lag', 'agreement_category']
  Granger significant: 1
  SHAP dependencies ≥ 0.01: 17

📊 Processing Granger causality...
  Found 1 significant relationships

🧠 Processing SHAP dependencies...
  Found 27 significant relationships

GRANGER NETWORK ANALYSIS - NorESM2-MM (MidLatitude_N)
Variables: 2
Relationships: 1
Density: 0.500

Most influenced variables (incoming relationships):
  T: 1

Most influential variables (outgoing relationships):
  SM: 1

Strongest causal relationships (lowest p-values):
  1. SM → T: p=3.29e-03, lag=2

SHAP NETWORK ANALYSIS - NorESM2-MM (MidLatitude_N)
Variables: 6
Relationships: 27
Density: 0.900

Most influenced variables (incoming relationships):
  T: 5
  Pr: 5
  Prw: 5
  E: 5
  q: 5
  SM: 2

Most influe


✅ Generated file:
  - combined_network.png

INTERPRETATION GUIDE:
📊 GRANGER CAUSALITY (Left Panel):
  • Edge thickness → Statistical significance (p-value)
  • Edge colors → Source variable identity
  • Shows temporal causal relationships

🧠 SHAP DEPENDENCIES (Right Panel):
  • Edge thickness → Dependency strength
  • Solid lines → Upward effect on target
  • Dashed lines → Downward effect on target
  • Shows feature importance relationships

✅ ANALYSIS COMPLETE!

To customize:
1. Edit the filepath to your CSV file
2. Change model_name and region as needed
3. Uncomment examples 2-4 for batch processing
4. Modify VARIABLE_MAPPING and colors as needed
